In [ ]:
# !pip install padelpy
# import padelpy

In [ ]:
# !pip install padelpy
# import padelpy

In [7]:
import pandas as pd
import numpy as np
from padelpy import padeldescriptor, from_smiles
from tqdm import tqdm

In [8]:
#load mmp7 inh dataset
df = pd.read_csv('BindingDB_QSAR.padel.csv')
df.head(10)

,BindingDB Reactant_set_id,Ligand SMILES
0,50477794,COC(=O)c1ccc(NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC...
1,50125514,CC(C)C[C@H]([C@H](O)C(=O)NO)C(=O)N[C@H](C(=O)c...
2,50125422,CC(C)C[C@H]([C@H](CC=C)C(=O)NO)C(=O)N[C@H](C(=...
3,50477797,CC(C)C[C@H](CC(=O)NO)C(=O)N[C@@H](CC(C)C)c1nc(...
4,50477804,COC(=O)c1ccc2nc([nH]c2c1)[C@H](CC(C)C)NC(=O)[C...
5,50125427,CC(C)C[C@H]([C@H](O)C(=O)NO)C(=O)N[C@@H](Cc1cc...
6,50125502,CC(C)C[C@H]([C@H](CC=C)C(=O)NO)C(=O)N[C@@H](Cc...
7,50125443,CC(C)C[C@@H]1[C@H](CCCCOc2ccc(C[C@H](NC1=O)C(=...
8,50125423,CNC(=O)[C@@H]1Cc2ccc(OCCCC[C@@H]([C@@H](CC(C)C...
9,50123108,CC(C)C[C@@H]1[C@H](CCCCOc2ccc(C[C@H](NC1=O)C(=...


## **Calculate molecular descriptors from padel**

 **Descriptor Info**
*   It will provide the following descriptors/fingerprint:
1.   2D - 1444
2.   3D - 431
3.   Fingerprints - PubChem fingerprint 881 bits


**Calculate molecular descriptors using "from_smiles" function**


> The "from_smiles" function accepts a SMILES string or list of SMILES strings as an argument, and returns a Python dictionary with descriptor/fingerprint names/values as keys/values respectively - if multiple SMILES strings are supplied, "from_smiles" returns a list of dictionaries.



In [25]:
# descriptors = from_smiles('CC(C)C[C@H]([C@H](O)C(=O)NO)C(=O)N[C@H](C(=O)c1c[nH]c2ccccc12)C(C)(C)C', descriptors=True, fingerprints=False)
# descriptors

In [6]:
df.head()

,BindingDB Reactant_set_id,Ligand SMILES
0,50477794,COC(=O)c1ccc(NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC...
1,50125514,CC(C)C[C@H]([C@H](O)C(=O)NO)C(=O)N[C@H](C(=O)c...
2,50125422,CC(C)C[C@H]([C@H](CC=C)C(=O)NO)C(=O)N[C@H](C(=...
3,50477797,CC(C)C[C@H](CC(=O)NO)C(=O)N[C@@H](CC(C)C)c1nc(...
4,50477804,COC(=O)c1ccc2nc([nH]c2c1)[C@H](CC(C)C)NC(=O)[C...


**Generate descriptors for the whole dataset**

In [12]:
# Load your dataset
smiles_list = df["Ligand SMILES"].tolist()

# Prepare results storage
results = []

# Process each SMILES with progress bar
for smile in tqdm(smiles_list, desc="Processing SMILES", unit="molecule"):
    try:
        result = from_smiles([smile], descriptors=True, fingerprints=False, timeout=900, maxruntime=-1, threads = 4)
        results.append(result)
    except RuntimeError as e:
        print(f"Failed for {smile}: {e}")

Processing SMILES: 100%|████████████████| 499/499 [21:52<00:00,  2.63s/molecule]


In [ ]:
#flatten the nested list and convert to DataFrame
df_results = [item[0] for item in results]  # Extract dictionaries from nested list
df_results = pd.DataFrame(df_results)

In [21]:
#Concatenate with the original SMILES column if needed
final_df = pd.concat([df, df_results], axis=1)
final_df.head()

,BindingDB Reactant_set_id,Ligand SMILES,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,50477794,COC(=O)c1ccc(NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC...,0,-1.1567000000000018,1.3379548900000042,86.2239,68.83616899999996,6,6,64,...,,,,,,,,,,
1,50125514,CC(C)C[C@H]([C@H](O)C(=O)NO)C(=O)N[C@H](C(=O)c...,0,-1.1418000000000017,1.3037072400000038,77.1854,66.70058299999995,9,10,61,...,0.788622059573347,0.13005859990948243,0.5372968518974188,0.4217061184598497,0.4233119982024924,20.909840696008253,77.50781230990526,174.67014052712517,0.6829330893600204,1.3823149685597609
2,50125422,CC(C)C[C@H]([C@H](CC=C)C(=O)NO)C(=O)N[C@H](C(=...,0,0.10079999999999734,0.010160639999999463,89.9322,73.84575499999994,9,10,67,...,,,,,,,,,,
3,50477797,CC(C)C[C@H](CC(=O)NO)C(=O)N[C@@H](CC(C)C)c1nc(...,0,-0.8468000000000013,0.7170702400000023,70.16910000000001,66.86337599999996,11,12,61,...,0.7462406768009733,0.15787353647043015,0.5109887164381968,0.33506282035391466,0.3456186032756011,24.702461282635095,124.79031954017974,319.772597363424,0.61936101520146,1.1916701400677125
4,50477804,COC(=O)c1ccc2nc([nH]c2c1)[C@H](CC(C)C)NC(=O)[C...,0,-0.835600000000001,0.6982273600000017,82.1575,68.46737599999996,9,10,63,...,0.7558620349884261,0.16345959888279857,0.4519015227118044,0.3614656233942535,0.386535986156786,28.061830967903692,155.69963761200074,404.0328465204066,0.6337930524826391,1.199903132262844


In [22]:
# save descriptors to a CSV file
final_df.to_csv("MMP7_inh_decriptor_results.csv", index=False)

In [61]:
#load c.arabica compound dataset
df = pd.read_csv("Coffea_arabica_Docked_Compounds_SMILES.csv", encoding="latin1")
df.head(10)

,Compound,SMILES
0,(+)-Gamma-cadinene,CC1=C[C@@H]2[C@@H](CC1)C(=C)CC[C@H]2C(C)C
1,(E) beta-OCIMENE,CC(=CC/C=C(\C)/C=C)C
2,(Z) beta-OCIMENE,CC(=CC/C=C(/C)\C=C)C
3,16-O-Methylcafestol,C[C@@]12CCC3=C([C@H]1CC[C@]45[C@H]2CC[C@H](C4)...
4,1-Hexanol,CCCCCCO
5,1-Octadecanesulphonyl chloride,CCCCCCCCCCCCCCCCCCS(=O)(=O)Cl
6,1-Octen-3-Ol,CCCCCC(C=C)O
7,1-Pentanol,CCCCCO
8,1-Penten-3-Ol,CCC(C=C)O
9,2-(2-Furfuryl)furan,C1=COC(=C1)CC2=CC=CO2


In [62]:
# Load your dataset
c_arabica_smiles_list = df["SMILES"].str.replace("\s+", "", regex=True).tolist()

# Prepare results storage
results = []

# Process each SMILES with progress bar
for smile in tqdm(c_arabica_smiles_list, desc="Processing SMILES", unit="molecule"):
    try:
        result = from_smiles([smile], descriptors=True, fingerprints=False, timeout=900, maxruntime=-1, threads = 4)
        results.append(result)
    except RuntimeError as e:
        print(f"Failed for {smile}: {e}")

Processing SMILES: 100%|████████████████| 161/161 [08:20<00:00,  3.11s/molecule]


In [68]:
#flatten the nested list and convert to DataFrame
df_results = [item[0] for item in results]  # Extract dictionaries from nested list
df_results = pd.DataFrame(df_results)
df_results.head()

,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,0,2.260300000000001,5.108956090000004,66.3785,42.403031999999975,0,0,39,15,24,...,0.5221891320495062,0.3941873010805373,0.5000682602916771,0.49017072751230767,0.41256027903151027,9.788662267008236,27.06577966132672,52.999112198748925,0.37456464969506525,1.402799266835495
1,0,3.2402999999999995,10.499544089999997,49.715900000000005,28.268687999999976,0,0,26,10,16,...,,,,,,,,,,
2,0,3.2402999999999995,10.499544089999997,49.715900000000005,28.268687999999976,0,0,26,10,16,...,,,,,,,,,,
3,0,-0.04319999999999863,0.0018662399999998815,94.65369999999999,59.36978999999996,5,5,54,24,30,...,0.6902549132625382,0.19154287006105156,0.4869867583239244,0.5028920888163021,0.4725485949714675,12.535594703394374,37.15505639200681,80.47542334358315,0.5353823698938074,1.4624274421116938
4,0,-1.5493999999999994,2.4006403599999984,24.8219,20.69710199999999,0,0,21,7,14,...,0.8367404818309013,0.10316303032741277,0.5998155002378469,0.6050736329096743,0.5349752296963706,7.613885163390105,8.278617805478474,18.182225881578226,0.7551107227463519,1.7398643628438917


In [69]:
#Concatenate with the original SMILES column if needed
final_df = pd.concat([df, df_results], axis=1)
final_df.head()

,Compound,SMILES,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,(+)-Gamma-cadinene,CC1=C[C@@H]2[C@@H](CC1)C(=C)CC[C@H]2C(C)C,0,2.260300000000001,5.108956090000004,66.3785,42.403031999999975,0,0,39,...,0.5221891320495062,0.3941873010805373,0.5000682602916771,0.49017072751230767,0.41256027903151027,9.788662267008236,27.06577966132672,52.999112198748925,0.37456464969506525,1.402799266835495
1,(E) beta-OCIMENE,CC(=CC/C=C(\C)/C=C)C,0,3.2402999999999995,10.499544089999997,49.715900000000005,28.268687999999976,0,0,26,...,,,,,,,,,,
2,(Z) beta-OCIMENE,CC(=CC/C=C(/C)\C=C)C,0,3.2402999999999995,10.499544089999997,49.715900000000005,28.268687999999976,0,0,26,...,,,,,,,,,,
3,16-O-Methylcafestol,C[C@@]12CCC3=C([C@H]1CC[C@]45[C@H]2CC[C@H](C4)...,0,-0.04319999999999863,0.0018662399999998815,94.65369999999999,59.36978999999996,5,5,54,...,0.6902549132625382,0.19154287006105156,0.4869867583239244,0.5028920888163021,0.4725485949714675,12.535594703394374,37.15505639200681,80.47542334358315,0.5353823698938074,1.4624274421116938
4,1-Hexanol,CCCCCCO,0,-1.5493999999999994,2.4006403599999984,24.8219,20.69710199999999,0,0,21,...,0.8367404818309013,0.10316303032741277,0.5998155002378469,0.6050736329096743,0.5349752296963706,7.613885163390105,8.278617805478474,18.182225881578226,0.7551107227463519,1.7398643628438917


In [70]:
# save descriptors to a CSV file
final_df.to_csv("Coffea_arabica_Docked_Compounds_desc.csv", index=False)